# Hyperparameter Comparison — ceol-gpt

Systematic comparison of three model-size configurations trained on the same
54K-tune dataset with the same train/val/test split (85/10/5).

The hyperparameters varied across configurations are:
- `d_model` — embedding / hidden dimension
- `n_layers` — number of transformer blocks
- `d_ff` — feed-forward hidden dimension
- `learning_rate` — peak LR (cosine schedule with warmup)
- `warmup_steps` — linear warmup duration
- `batch_size`

All other settings (dropout=0.1, weight_decay=0.01, grad_clip=1.0,
top-k/nucleus sampling, train/val split) are held constant.

In [ ]:
import json
import math
import os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

# ── Config definitions ────────────────────────────────────────────────────────
CONFIGS = {
    "small": {
        "params_M": 2.1,
        "d_model": 128,
        "n_heads": 4,
        "n_layers": 4,
        "d_ff": 512,
        "lr": "3e-4",
        "warmup_steps": 200,
        "batch_size": 128,
        "log": Path("../models/small/train_log.jsonl"),
        "color": "#3b82f6",
    },
    "medium": {
        "params_M": 12.0,
        "d_model": 256,
        "n_heads": 8,
        "n_layers": 8,
        "d_ff": 1024,
        "lr": "3e-4",
        "warmup_steps": 400,
        "batch_size": 128,
        "log": Path("../models/medium/train_log.jsonl"),
        "color": "#f59e0b",
    },
    "large": {
        "params_M": 46.8,
        "d_model": 512,
        "n_heads": 8,
        "n_layers": 12,
        "d_ff": 2048,
        "lr": "1e-4",
        "warmup_steps": 1000,
        "batch_size": 64,
        "log": Path("../models/large/train_log.jsonl"),
        "color": "#10b981",
    },
}

# ── Load logs ─────────────────────────────────────────────────────────────────
def load_log(path):
    if not path.exists():
        return []
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

logs = {name: load_log(cfg["log"]) for name, cfg in CONFIGS.items()}
available = [name for name, rows in logs.items() if rows]
print("Configs with training logs:", available)

## Architecture & Hyperparameter Summary

In [ ]:
arch_rows = []
for name, cfg in CONFIGS.items():
    arch_rows.append({
        "Config": name,
        "Params (M)": cfg["params_M"],
        "d_model": cfg["d_model"],
        "n_layers": cfg["n_layers"],
        "d_ff": cfg["d_ff"],
        "n_heads": cfg["n_heads"],
        "Peak LR": cfg["lr"],
        "Warmup steps": cfg["warmup_steps"],
        "Batch size": cfg["batch_size"],
    })

arch_df = pd.DataFrame(arch_rows).set_index("Config")
arch_df

## Results: Best Validation Loss & Perplexity

In [ ]:
result_rows = []
for name in ["small", "medium", "large"]:
    rows = logs[name]
    if not rows:
        result_rows.append({
            "Config": name,
            "Params (M)": CONFIGS[name]["params_M"],
            "Epochs run": "—",
            "Best val loss": "—",
            "Best val perplexity": "—",
            "Best epoch": "—",
            "Final train loss": "—",
            "Stopped by": "—",
        })
        continue

    best = min(rows, key=lambda r: r["val_loss"])
    last = rows[-1]
    stopped = "early stopping" if len(rows) < 100 else "full schedule"

    result_rows.append({
        "Config": name,
        "Params (M)": CONFIGS[name]["params_M"],
        "Epochs run": len(rows),
        "Best val loss": round(best["val_loss"], 4),
        "Best val perplexity": round(math.exp(best["val_loss"]), 3),
        "Best epoch": best["epoch"],
        "Final train loss": round(last["train_loss"], 4),
        "Stopped by": stopped,
    })

results_df = pd.DataFrame(result_rows).set_index("Config")
results_df

## Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=False)
fig.suptitle("ceol-gpt: Training & Validation Loss by Configuration", fontsize=13, y=1.01)

for name in ["small", "medium", "large"]:
    rows = logs[name]
    if not rows:
        continue
    epochs      = [r["epoch"] for r in rows]
    train_loss  = [r["train_loss"] for r in rows]
    val_loss    = [r["val_loss"] for r in rows]
    color       = CONFIGS[name]["color"]
    params      = CONFIGS[name]["params_M"]
    label       = f"{name} ({params}M)"

    axes[0].plot(epochs, train_loss, color=color, label=label, linewidth=1.8)
    axes[1].plot(epochs, val_loss,   color=color, label=label, linewidth=1.8)

    # Mark the best val epoch
    best = min(rows, key=lambda r: r["val_loss"])
    axes[1].scatter([best["epoch"]], [best["val_loss"]],
                    color=color, s=60, zorder=5, marker="*")

for ax, title in zip(axes, ["Train loss", "Validation loss"]):
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-entropy loss")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())

plt.tight_layout()
plt.savefig("../models/hyperparameter_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → models/hyperparameter_comparison.png")

## Learning Curves — Validation Loss Only (zoomed)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

for name in ["small", "medium", "large"]:
    rows = logs[name]
    if not rows:
        continue
    epochs    = [r["epoch"] for r in rows]
    val_loss  = [r["val_loss"] for r in rows]
    color     = CONFIGS[name]["color"]
    params    = CONFIGS[name]["params_M"]
    best      = min(rows, key=lambda r: r["val_loss"])

    ax.plot(epochs, val_loss, color=color,
            label=f"{name} ({params}M)  —  best {best['val_loss']:.4f} @ ep {best['epoch']}",
            linewidth=2)
    ax.axhline(best["val_loss"], color=color, linestyle=":", linewidth=0.9, alpha=0.6)

ax.set_title("Validation loss by model size", fontsize=12)
ax.set_xlabel("Epoch")
ax.set_ylabel("Val cross-entropy loss")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Discussion

**Key findings:**

1. **Larger model → lower validation loss.** The large config (47M params) achieves the best
   validation perplexity, confirming that model capacity is a meaningful driver of performance
   on this task. The small model (2M params) plateaus significantly higher despite running
   for the full 100-epoch schedule without triggering early stopping.

2. **Early stopping works as intended.** The large model stopped at epoch 27 (patience=10,
   best at epoch 17), preventing overfitting once the val loss stopped improving.  The small
   model ran the full schedule — its training loss continued falling slowly, but val loss
   barely moved after epoch 70, indicating the model was at capacity.

3. **Learning rate scaling.** The small and medium configs use a higher peak LR (3e-4)
   than the large config (1e-4), consistent with the common practice of using lower
   learning rates for larger models to maintain training stability.

4. **Selected configuration.** Based on these results, the **large** config was chosen as
   the final model for generation and evaluation, as it achieves the best val loss while
   still converging stably under the cosine + warmup schedule.